In [1]:
# If IMAGINE_Concrete is not importable, run from this folder or uncomment:
# import sys; from pathlib import Path
# sys.path.insert(0, str(Path.cwd().parent / '1-IMAGINE-Concrete'))

import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
from IMAGINE_Concrete import run_historical, run_projection, load_shared_data, DATA

In [2]:
shared = load_shared_data()
economies_142 = shared["df_cement_production"].columns.tolist()

In [ ]:
# run_historical("CHN", shared)

In [ ]:
for economy in tqdm(economies_142):
    run_historical(economy, shared)

## CE scenarios (income-group-differentiated, ore-sand-supply-capped)

Every lever is read **per economy** from `CE-parameters-142.csv` (HIC/UMIC/LMIC/LIC targets); sand substitution is additionally capped by the national ore-sand supply potential `oresand-supply-cap-142.csv` (kt/yr), and clinker uses the absolute 2050 clinker-to-cement target (`clinker_target`, ramped from the base-year ratio, reduction-only).

> `run_projection` uses SciPy; if this notebook runs **outside** the activated `wty` environment it can crash at the logistic-fit step. Use the `wty` Jupyter kernel (or `conda run -n wty`).

In [3]:
# income-group-differentiated CE targets + per-economy ore-sand supply cap (kt/yr)
ce  = pd.read_csv(DATA / 'CE-parameters-142.csv').set_index('economy')
oss = pd.read_csv(DATA / 'oresand-supply-cap-142.csv').set_index('economy')

In [4]:
# run_projection kwarg  ->  CE-parameters-142.csv column
LEVER_COL = {
    'more_intensive_use': 'intensive_use',
    'lifetime_extension': 'lifetime_ext',
    'reuse_rate':         'reuse',
    'recycle_rate':       'recycling',
    'sand_sub':           'sand_substitution',
    'clinker_target':     'clinker_ratio_target',
}
ALL_LEVERS = set(LEVER_COL)

In [5]:
def ce_kwargs(economy, levers=ALL_LEVERS, groups=None):
    """Per-economy run_projection kwargs.
    A lever is ON only if it is in `levers` AND (groups is None or the economy's
    income group is in `groups`). OFF levers fall back to neutral values."""
    r = ce.loc[economy]
    group_on = (groups is None) or (r['income_group'] in groups)
    kw = {}
    for arg, col in LEVER_COL.items():
        on = group_on and (arg in levers)
        if arg == 'clinker_target':
            kw[arg] = float(r[col]) if on else None          # None -> clinker held at base
        else:
            kw[arg] = float(r[col]) if on else 0.0
    sand_on = group_on and ('sand_sub' in levers)
    kw['oresand_supply'] = float(oss.loc[economy, 'oresand_supply_kt_yr']) if sand_on else np.inf
    return kw

In [6]:
def run_scenario(scenario, levers=ALL_LEVERS, groups=None, ambition=1.0, n_jobs=12):
    """Run all 142 economies for one scenario, each with its differentiated kwargs.
    `ambition` (phi in [0,1]) globally scales every economy's CE target inside run_projection:
    0 -> BAU, 1 -> full CE. Preserves cross-country differentiation."""
    Parallel(n_jobs=n_jobs, backend='loky', verbose=5)(
        delayed(run_projection)(scenario, e, ambition=ambition, **ce_kwargs(e, levers, groups))
        for e in economies_142
    )

In [ ]:
# quick single-economy check before the full sweeps (CHN)
run_projection('CE_test', 'CHN', **ce_kwargs('CHN'))
print('CHN CE test done')

In [10]:
# business-as-usual: no CE levers
run_scenario('BAU', levers=set())

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  4.8min
[Parallel(n_jobs=12)]: Done 142 out of 142 | elapsed: 13.8min finished


In [7]:
# full circular-economy scenario: all levers, all economies (differentiated + supply-capped)
run_scenario('CE')

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:   43.6s
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:  5.8min
[Parallel(n_jobs=8)]: Done 142 out of 142 | elapsed: 14.4min finished


In [7]:
# single-lever isolation scenarios (each income-group-differentiated)
ISO = {
    'MIU': 'more_intensive_use',
    'LE':  'lifetime_extension',
    'RE':  'reuse_rate',
    'RC':  'recycle_rate',
    'SS':  'sand_sub',
    'CCR': 'clinker_target',
}
for scen, lever in ISO.items():
    print(scen)
    run_scenario(scen, levers={lever})

MIU


[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  4.8min
[Parallel(n_jobs=12)]: Done 142 out of 142 | elapsed: 13.9min finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


LE


[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  5.0min
[Parallel(n_jobs=12)]: Done 142 out of 142 | elapsed: 14.1min finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


RE


...

[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  5.0min
[Parallel(n_jobs=12)]: Done 142 out of 142 | elapsed: 13.7min finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


SS


[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  4.8min
[Parallel(n_jobs=12)]: Done 142 out of 142 | elapsed: 13.8min finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


CCR


[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  4.9min
[Parallel(n_jobs=12)]: Done 142 out of 142 | elapsed: 13.5min finished


## Phased adoption by income group (trade-off frontier)

CE measures roll out in capacity order — **HIC first, LIC last** — until all economies reach full potential. Each stage activates *all* levers (and the supply cap) for the included groups while the rest stay at BAU. This is a **capacity-diffusion frontier**, not a time forecast.

Frontier points: `BAU` (0) → `HIC` (50) → `HIC+UMIC` (85) → `HIC+UMIC+LMIC` (120) → **`CE`** (142; = phase 4, reuse the full-CE run above).

In [ ]:
# PHASES = [
#     ('HIC',          {'HIC'}),
#     ('HIC+UMIC',     {'HIC', 'UMIC'}),
#     ('HIC+UMIC+LMIC',{'HIC', 'UMIC', 'LMIC'}),
#     # phase 4 (all groups) == the 'CE' scenario above
# ]
# for scen, groups in PHASES:
#     print(scen, sorted(groups))
#     run_scenario(scen, groups=groups)

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


HIC ['HIC']


[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:   40.2s
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:  5.6min
[Parallel(n_jobs=8)]: Done 142 out of 142 | elapsed: 14.0min finished
[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


HIC+UMIC ['HIC', 'UMIC']


[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:   39.3s
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:  5.5min
[Parallel(n_jobs=8)]: Done 142 out of 142 | elapsed: 13.2min finished
[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


HIC+UMIC+LMIC ['HIC', 'LMIC', 'UMIC']


[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:   41.0s
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:  5.2min
[Parallel(n_jobs=8)]: Done 142 out of 142 | elapsed: 12.8min finished


## CE-ambition sweep (proportional knob φ) — the demand axis

A single global multiplier **φ ∈ [0, 1]** scales *every* economy's CE target (`target_used = φ · target`), implemented inside `run_projection(..., ambition=φ)`. This gives a continuous, evenly-spaced **demand axis** for the cost–biodiversity trade-off surface — replacing the lumpy income-group phasing (where UMIC alone drove ~⅔ of the reduction and LIC ≈ 0).

- **φ = 0 ≡ BAU**, **φ = 1 ≡ CE** (both already run above) → only φ ∈ {0.2, 0.4, 0.6, 0.8} are new.
- Cross-country differentiation is preserved (each economy keeps its income-group target shape; φ scales all uniformly).
- The 6 levers scale linearly; **clinker** (absolute k-factor) is scaled inside its own ramp: `k = base − φ·max(base−target, 0)·ramp`.
- Keep the income-group / single-lever runs above for the **contribution-decomposition** figure (they answer "who/which lever", a separate result).

In [ ]:
# CE-ambition sweep (proportional knob phi): each economy's full CE target x phi.
# phi=0 == BAU and phi=1 == CE are already run above, so only the intermediate phi are needed.
# All levers, all groups, ore-sand supply-capped. Output names CE20..CE80 (= phi*100).
# Demand axis for the trade-off surface then becomes:  [BAU, CE20, CE40, CE60, CE80, CE]
# (= phi 0, .2, .4, .6, .8, 1.0). Downstream: re-run 1-downscale-extraction-demand -> 5-2 -> 5-3 -> 4/6.
for phi in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    scen = f'CE{int(round(phi * 100))}'
    print(scen, f'(phi={phi})')
    run_scenario(scen, ambition=phi)

CE20 (phi=0.2)


[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  5.4min
[Parallel(n_jobs=12)]: Done 142 out of 142 | elapsed: 15.8min finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


CE40 (phi=0.4)


[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  5.3min
[Parallel(n_jobs=12)]: Done 142 out of 142 | elapsed: 14.3min finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


CE60 (phi=0.6)


[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  4.7min
[Parallel(n_jobs=12)]: Done 142 out of 142 | elapsed: 13.4min finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.


CE80 (phi=0.8)


[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  5.1min
[Parallel(n_jobs=12)]: Done 142 out of 142 | elapsed: 13.8min finished


## Supplementary Figure 8-10 — cumulative-measure ladder (n_sim=10,000)

SI Fig 8-10 decomposes future extraction by CE measure, adding levers one at a time:
**BAU → +more intensive use → +lifetime extension → +component reuse → +end-of-life recycling → +aggregate substitution → +clinker substitution (= CE)**. The figure plots the **per-year median over n_sim=10,000** (a single n_sim=1 draw gives jagged year-to-year lines).

The last measure, *clinker substitution*, completes the full lever set, so the "all measures" rung **is** the `CE` scenario — the clinker wedge is `wedge_SS − CE`. Endpoints therefore reuse the production `BAU`/`CE` runs above; only the **5 intermediate** cumulative rungs are computed here, on the production `historical` basis, and each folder is **pruned** to the 3 `<material>_extraction.npz` the figure needs (a full 10k folder is ~180 MB → a few MB). Idempotent (re-run-safe). Same as `1-IMAGINE-Concrete/run-wedge-scenarios.py`.

In [ ]:
# === SI Fig 8-10 cumulative-measure ladder at n_sim=10,000 (smooth per-year median) ===
# BAU and CE reuse the production scenarios above (clinker substitution == the SS->CE step,
# so the "all measures" rung IS the CE scenario). Only the 5 INTERMEDIATE cumulative rungs
# are computed here, on the production 10,000-run 'historical', and each folder is pruned to
# the 3 <material>_extraction.npz the figure needs (a full 10k folder is ~180 MB -> a few MB).
# Idempotent (re-run-safe). Mirrors 1-IMAGINE-Concrete/run-wedge-scenarios.py.
from IMAGINE_Concrete import RESULTS
KEEP = {'gravel_extraction.npz', 'sand_extraction.npz', 'limestone_extraction.npz'}
WEDGE_LADDER = [
    ('wedge_MIU', {'more_intensive_use'}),
    ('wedge_LE',  {'more_intensive_use', 'lifetime_extension'}),
    ('wedge_RE',  {'more_intensive_use', 'lifetime_extension', 'reuse_rate'}),
    ('wedge_RC',  {'more_intensive_use', 'lifetime_extension', 'reuse_rate', 'recycle_rate'}),
    ('wedge_SS',  {'more_intensive_use', 'lifetime_extension', 'reuse_rate', 'recycle_rate', 'sand_sub'}),
]

def _run_wedge(scen, economy, levers):
    d = RESULTS / 'material-flows' / economy / scen
    if d.exists() and {p.name for p in d.glob('*.npz')} == KEEP:
        return                                       # idempotent: already done & pruned
    run_projection(scen, economy, n_sim=10000, hist_subdir='historical', **ce_kwargs(economy, levers))
    for f in d.glob('*.npz'):
        if f.name not in KEEP:
            f.unlink()

for scen, levers in WEDGE_LADDER:
    print(scen, sorted(levers))
    Parallel(n_jobs=12, backend='loky', verbose=5)(
        delayed(_run_wedge)(scen, e, levers) for e in economies_142)
print('SI Fig 8-10 ladder done (n_sim=10,000, pruned). Draw: 9-draw-figures/Supplementary-Figure-8-10.ipynb')